# Exercise 2 — score_headline

`score_headline` is the core LLM call: build the prompt, call the model (or the injected mock), parse the response. In production it calls Ollama; in tests it calls `_mock_llm`. This injection pattern keeps the exercises fast and deterministic — no network, no model download required.

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]
def parse_score(text):
    """Extract and clamp a float from LLM output. Returns 0.0 if not found."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return 0.0
    return max(-1.0, min(1.0, float(matches[0])))

def build_sentiment_prompt(headline):
    return [
        {
            "role": "system",
            "content": (
                "You are a financial news sentiment analyzer. "
                "Score the sentiment from -1.0 (very bearish) to 1.0 (very bullish). "
                "Reply with ONLY a single decimal number. No explanation."
            ),
        },
        {"role": "user", "content": f"Headline: {headline}"},
    ]

def score_headline(headline, llm_fn=None):
    """Score one headline for financial sentiment.

    Steps:
      1. messages  = build_sentiment_prompt(headline)
      2. if llm_fn is not None:
             response = llm_fn(messages)
         else:
             import ollama
             response = ollama.chat(model="llama3.2",
                                    messages=messages)["message"]["content"]
      3. return parse_score(response)

    Args:
        headline : str — a financial news headline
        llm_fn   : callable(messages) -> str  (inject _mock_llm for testing)

    Returns:
        float in [-1.0, 1.0]
    """
    # TODO: implement the 3 steps
    return 0.0


### Checks

In [ ]:
checks = 0

# 1 — returns a float in [-1.0, 1.0]
try:
    s = score_headline("Markets rally strongly", llm_fn=_mock_llm)
    assert isinstance(s, float), f"expected float, got {type(s)}"
    assert -1.0 <= s <= 1.0,     f"out of range: {s}"
    checks += 1; print("✅ 1 score_headline returns float in [-1.0, 1.0]")
except Exception as e:
    print("❌ 1:", e)

# 2 — bullish headlines score positively with mock
try:
    s = score_headline("S&P 500 surges on strong jobs data", llm_fn=_mock_llm)
    assert s > 0, f"expected positive score for bullish headline, got {s}"
    checks += 1; print("✅ 2 bullish headline → positive score")
except Exception as e:
    print("❌ 2:", e)

# 3 — bearish headlines score negatively with mock
try:
    s = score_headline("Markets crash amid recession fears", llm_fn=_mock_llm)
    assert s < 0, f"expected negative score for bearish headline, got {s}"
    checks += 1; print("✅ 3 bearish headline → negative score")
except Exception as e:
    print("❌ 3:", e)

# 4 — llm_fn is called with a list of messages (not a plain string)
try:
    received_args = []
    def _capturing_llm(messages):
        received_args.append(messages)
        return "0.5"
    score_headline("Any headline", llm_fn=_capturing_llm)
    assert len(received_args) == 1, "llm_fn should be called exactly once"
    assert isinstance(received_args[0], list), f"expected list of messages, got {type(received_args[0])}"
    checks += 1; print("✅ 4 llm_fn called with messages list")
except Exception as e:
    print("❌ 4:", e)

# 5 — returns 0.0 when llm returns non-number
try:
    def _bad_llm(messages): return "no numbers here at all"
    s = score_headline("Some headline", llm_fn=_bad_llm)
    assert s == 0.0, f"expected 0.0 for non-number response, got {s}"
    checks += 1; print("✅ 5 returns 0.0 when LLM returns unparseable output")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
